**27 | Final HARTH Research Evidence Synthesis and Reproducibility Report**

**Human Movement Intelligence | Reza Saadatyar (2026)**

This notebook consolidates the actual executed Stage 23–26 saved evidence and frozen protocol without downloading HARTH data, retraining models, loading model weights, or accessing either original Test recording. Stage 23/24 metrics are Development-only; Stage 26 is a qualified same-study held-out evaluation of two historically inspected participants, not blind external validation. All reported notebook tables have stored display precision, usually four decimals. No missing metrics are interpolated.

**01. Evidence Scope and Notebook Integrity**

Read the executed benchmark, robustness, final model-selection and qualified held-out notebooks directly from the checkout. Check source fingerprints, execution counts, saved errors and their explicit final Quality Gates. Fail rather than fabricate when an expected result is missing.

In [ ]:
# ==================== 01. Evidence Sources and Integrity ====================
# 1. Find the repository without assuming a local machine-specific path.
import sys
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
root = Path.cwd().resolve()
while not (root/"studies/wearable_activity_recognition/notebooks/26_final_development_training_and_inference.ipynb").is_file():
    if root.parent == root:
        raise RuntimeError("Start inside the human-movement-intelligence repository.")
    root = root.parent
sys.path.insert(0, str(root/"code/python/src"))
from core.harth_selection_evidence import load_selection_evidence
from core.harth_study_report import load_final_harth_evidence, _load_executed_notebook
base = root/"studies/wearable_activity_recognition"
notebook_dir = base/"notebooks"
stage23 = notebook_dir/"23_unified_matched_model_benchmark.ipynb"
stage24 = notebook_dir/"24_multiseed_and_imbalance_robustness.ipynb"
stage25 = notebook_dir/"25_model_selection_and_protocol_freeze.ipynb"
stage26 = notebook_dir/"26_final_development_training_and_inference.ipynb"
manifest_pending = base/"experiments/stage_25_protocol_proposal_reconciled.json"
manifest_approved = base/"experiments/stage_25_protocol_approved_qualified_test.json"
development = load_selection_evidence(stage23, stage24)
final = load_final_harth_evidence(
    stage26, manifest_pending, manifest_approved, development["development_sha256"]
)
n25, h25 = _load_executed_notebook(stage25,"Stage 25 development selection audit: PASSED")
assert final["approved_manifest"]["stage23_saved_notebook_sha256"] == development["notebook23_sha256"]
assert final["approved_manifest"]["stage24_saved_notebook_sha256"] == development["notebook24_sha256"]
print("="*76)
print("STAGE 27 | SOURCE-GROUNDED HARTH REPORT")
print("="*76)
print("Executed Stages 23, 24, 25, 26: VERIFIED")
print("Development fingerprint:",development["development_sha256"])
print("Stage 26 notebook SHA256:",final["notebook26_sha256"])
print("Original Test CSVs read by Stage 27: NO")
print("Models trained or scored by Stage 27: NO")
print("="*76)

**02. Dataset, Cohort and Experimental Design**

The source is pinned HARTH v2.0 from the NTNU AI Lab source repository, not the older UCI HARTH release. Six lower-back/right-thigh accelerometer channels sampled at nominal 50 Hz are windowed to 100 samples with stride 50, strict annotation purity and continuity. Eight original documented class codes are used; undocumented codes are not given invented activity names. A label-independent cap of 500 windows per person yields 14,201 Development windows from 29 people. Test uses 500 eligible windows each from S023 and S032, who had been inspected in earlier exploration. Correlated overlapping windows are not independent observations.

In [ ]:
# ==================== 02. Frozen Study Configuration ====================
# 1. Summarise the actual approved configuration, never a guessed protocol.
source = final["approved_manifest"]["source"]
primary = final["approved_manifest"]["primary_model"]
reference = final["approved_manifest"]["reference_model"]
protocol_rows = [
    ("Source dataset",source["dataset"]),
    ("Pinned revision",source["revision"]),
    ("Development people",source["development_participants"]),
    ("Development windows",source["expected_development_windows"]),
    ("Reserved Test people",", ".join(source["reserved_test_people"])),
    ("Test cap per person",source["test_cap_per_person"]),
    ("Sampling frequency (Hz)",source["sampling_rate_hz"]),
    ("Window / stride (samples)",f'{source["window_size"]} / {source["step_size"]}'),
    ("Classification codes",str(source["task_codes"])),
    ("Primary network",primary["architecture"]),
    ("Fixed optimiser updates",primary["optimizer_steps"]),
    ("Class weighting",primary["training_loss"]),
    ("Random Forest trees",reference["n_estimators"]),
]
display(pd.DataFrame(protocol_rows,columns=["Protocol component","Recorded setting"]))
display(final["coverage"])
assert final["coverage"]["sampled_windows"].sum()==1000

**03. Seven-Model Matched Development Comparison (Stage 23)**

Seven predeclared classifiers were evaluated using the same 29 Development participants and three subject-disjoint folds. Compare saved mean Macro F1, observed variability, accuracy and computational cost. Architecture, features and training-loss choice can differ between systems; a ranking is descriptive, not a causal architecture-only experiment.

In [ ]:
# ==================== 03. Seven-Model Development Benchmark ====================
# 1. Display all seven saved results, including weaker baselines.
benchmark = development["stage23"].sort_values("macro_f1_mean",ascending=False)
assert len(benchmark)==7
display(benchmark.round(4))
# 2. Make a labelled figure using exactly the stored mean and fold SD.
fig,ax = plt.subplots(figsize=(10,5))
ax.barh(benchmark["model"][::-1],benchmark["macro_f1_mean"][::-1],
        xerr=benchmark["macro_f1_sd"][::-1],capsize=3)
ax.set_xlabel("Development participant-held-out Macro F1 (mean ± fold SD)")
ax.set_title("Stage 23 | Seven-model matched Development benchmark")
ax.set_xlim(0,1)
fig.tight_layout()
plt.show()

**04. Repeated-Seed Robustness and Training-Loss Comparisons (Stage 24)**

Stage 24 evaluated 36 model/fold/seed jobs using seeds 42, 123 and 2026 across three disjoint Development folds. Report all four selected model/loss conditions rather than only the winning configuration. Across-seed SD refers to variation among three seed-level fold averages; it is not a confidence interval or independent cohort replication.

In [ ]:
# ==================== 04. Robustness and Loss Ablations ====================
# 1. Preserve all four stored experimental conditions and true seed counts.
robust = development["stage24_summary"].sort_values(
    "mean_across_seed_fold_macro_f1",ascending=False
)
assert len(development["stage24_folds"])==36 and len(robust)==4
display(robust.round(4))
# 2. Present consistent labelled mean/SD comparison.
labels = robust["architecture"].astype(str)+" | "+robust["loss_method"].astype(str)
fig,ax=plt.subplots(figsize=(10,4))
ax.barh(labels[::-1],robust["mean_across_seed_fold_macro_f1"][::-1],
        xerr=robust["across_seed_sd"][::-1],capsize=3)
ax.set_xlim(0,1)
ax.set_xlabel("Development Macro F1 (seed/fold mean ± across-seed SD)")
ax.set_title("Stage 24 | Repeated-seed robustness and loss comparison")
fig.tight_layout()
plt.show()

**05. Fixed Model-Selection Rationale and Final Development Fit (Stages 25–26)**

The selected primary was a 1D-ResNet with inverse-square-root weighted cross-entropy, 400 Adam updates, batch size 128, learning rate 0.001 and seed 42. The reference was a 120-tree Random Forest with 30 statistical features and fixed hyperparameters. Final fitting used 29 Development people only, without Test tuning. The saved Stage 26 Development inference reload check passed. Detailed model weights remain ignored local artefacts, not necessarily present in a new runtime. Stage 25 was selection on Development, not independent confirmation.

In [ ]:
# ==================== 05. Frozen Decision and Test Boundary ====================
# 1. Display the explicit pre-Test decision and integrity constraints.
decision = pd.DataFrame([
    {"Role":"Preselected primary","Model":"1D-ResNet Weighted CE",
     "Selection basis":"Development Stage 23/24"},
    {"Role":"Predeclared reference","Model":"Random Forest",
     "Selection basis":"Development Stage 23/24"},
])
display(decision)
assert primary["architecture"]=="1D-ResNet" and primary["optimizer_steps"]==400
assert reference["n_estimators"]==120
assert final["approved_manifest"]["evaluation"]["use_test_for_tuning"] is False
print("Model choice frozen before qualified Test: YES")
print("Test-driven hyperparameter/model changes permitted: NO")

**06. Qualified Held-Out Evaluation (Stage 26)**

The original Test source participants were S023 and S032, with exactly 500 eligible windows each, analysed once under the approved frozen protocol. Scores below come from the saved executed Notebook 26 tables, not new inference. The **primary metric** is fixed eight-class Macro F1. Balanced accuracy is computed across true classes observed in the evaluated subset and is not interchangeable with the fixed-eight-class denominator. Differences in accuracy and Macro F1 reflect different error profiles.

In [ ]:
# ==================== 06. Qualified Test Summary and Participants ====================
# 1. Report both frozen models and each person without replacing any score.
display(final["summary"].round(4))
display(final["per_subject"].round(4))
# 2. Show the actual participant heterogeneity.
fig,ax = plt.subplots(figsize=(8,4))
pivot=final["per_subject"].pivot(index="subject_id",columns="model",
                                values="fixed_eight_class_macro_f1")
pivot.plot.bar(ax=ax,ylim=(0,1),rot=0)
ax.set_ylabel("Fixed eight-class Macro F1")
ax.set_xlabel("Historically inspected original Test participant")
ax.set_title("Stage 26 | Qualified participant-level descriptive evaluation")
ax.legend(title="Frozen model")
fig.tight_layout()
plt.show()

**07. Activity-Level Errors, Actual Support and Class Imbalance**

Present precision, recall, F1 and true support for all eight source classes. In the qualified Test subset, Shuffling, Stairs ascending and Stairs descending have extremely small counts. An observed high recall on three or six windows does not demonstrate reliable generalisation. The confusion matrices themselves remain in Notebook 26; this notebook does not infer their counts from rounded aggregate metrics.

In [ ]:
# ==================== 07. Eight-Class Performance and Support ====================
# 1. Keep all sixteen original displayed model/class rows.
per_class=final["per_class"].sort_values(["model","source_code"])
display(per_class.round(4))
support=per_class[per_class.model=="1D-ResNet Weighted CE"]
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(support["activity"],support["support"])
ax.set_ylabel("Original Test window count")
ax.set_title("Stage 26 | True activity support (1,000 selected windows)")
ax.tick_params(axis="x",rotation=40)
fig.tight_layout()
plt.show()
assert int(support["support"].sum())==1000
print("Rare-class evidence must be qualified; no support-based significance claim.")

**08. Source Quality, Evaluation Limitations and Claims Boundary**

The source reports undocumented HARTH codes 10 and 11 in S032. They were left uninterpreted and excluded after annotation-safe windowing. Participants S023 and S032 were descriptively inspected during earlier development exploration; they are not a historically blind or externally independent cohort. Only two people were evaluated, with overlapping source windows and participant-dependent class support. This study does not establish clinical efficacy, robustness to unseen data sources, medical deployment, population-wide performance, statistically significant superiority or causal benefit of the architecture. Development fold metrics and final qualified-Test metrics must remain separate.

In [ ]:
# ==================== 08. Research Interpretation Register ====================
# 1. Explicitly separate evidence, limitations and unsupported claims.
boundaries=pd.DataFrame([
    ("Supported","29-person Development, 3 subject-disjoint folds and Stage 24 3-seed robustness"),
    ("Supported","Frozen models assessed on identical windows from S023 and S032"),
    ("Supported","Stage 26 source-aligned class metrics and qualified same-study results"),
    ("Limitation","Only 2 originally reserved people; both historically explored"),
    ("Limitation","Overlapping windows, small rare-class support, different per-person class coverage"),
    ("Limitation","Previously undocumented codes 10/11 were excluded, not reinterpreted"),
    ("Not supported","Independent blind external validation or prospective clinical deployment"),
    ("Not supported","Test-selected architectures, clinical decisions, population-level certainty"),
],columns=["Evidence status","Interpretation"])
display(boundaries)

**09. Reproducible Report Export and Provenance**

Store a compact report in ignored local outputs, including hashes for source notebooks, the Development cohort fingerprint and both frozen manifests. No original HARTH source rows or personal data are exported. GitHub retains the executed notebooks, protocols and source code; Test prediction CSVs produced by Stage 26 stay local and are not required to regenerate these displayed results.

In [ ]:
# ==================== 09. Export Evidence-Only Tables and Manifest ====================
# 1. Export only the checked displayed evidence, not raw recordings.
out=root/"outputs/harth_stage27_report"
out.mkdir(parents=True,exist_ok=True)
benchmark.to_csv(out/"development_seven_model_benchmark.csv",index=False)
robust.to_csv(out/"development_multiseed_robustness.csv",index=False)
final["summary"].to_csv(out/"qualified_test_summary_saved_display.csv",index=False)
final["per_subject"].to_csv(out/"qualified_test_per_subject_saved_display.csv",index=False)
final["per_class"].to_csv(out/"qualified_test_per_class_saved_display.csv",index=False)
lineage={
  "study":"HARTH v2.0",
  "scope":"saved output synthesis, no retraining, no Test re-evaluation",
  "source_revision":source["revision"],
  "development_sha256":development["development_sha256"],
  "notebook23_sha256":development["notebook23_sha256"],
  "notebook24_sha256":development["notebook24_sha256"],
  "notebook25_sha256":h25,
  "notebook26_sha256":final["notebook26_sha256"],
  "approved_manifest_sha256":hashlib.sha256(manifest_approved.read_bytes()).hexdigest(),
  "qualified_test_subjects":["S023","S032"],
  "test_already_historically_inspected":True,
  "metric_precision":"saved displayed four-decimal tables; no reconstruction of full predictions",
}
(out/"report_provenance.json").write_text(json.dumps(lineage,indent=2)+"\n",encoding="utf-8")
print("Report-only files written to:",out)
print("Raw HARTH data opened or downloaded during Stage 27: NO")

**10. Final Evidence Quality Gate and Publication-Ready Boundary**

Passing means the executed evidence and its saved metrics are internally consistent, the original scientific selection and qualified-Test boundaries are retained, and the tables required for a responsible report are present. It does **not** mean this is a peer-reviewed manuscript, an external evaluation, statistically significant improvement or independently replicated code execution. Review all reported figures and wording before revising the public README or tagging the first release.

In [ ]:
# ==================== 10. Final Research Report Quality Gate ====================
# 1. Confirm all source-grounded result families exist and their provenance matches.
assert len(benchmark)==7 and len(robust)==4
assert len(final["summary"])==2 and len(final["per_subject"])==4
assert len(final["per_class"])==16
assert set(final["per_class"].source_code)==set(range(1,9))
assert final["coverage"]["sampled_windows"].tolist()==[500,500]
assert final["approved_manifest"]["status"]=="APPROVED"
assert all((out / filename).is_file() for filename in (
    "development_seven_model_benchmark.csv",
    "development_multiseed_robustness.csv",
    "qualified_test_summary_saved_display.csv",
    "qualified_test_per_subject_saved_display.csv",
    "qualified_test_per_class_saved_display.csv",
    "report_provenance.json",
))
# 2. Do not misrepresent these retrospective source summaries as external validation.
print("="*76)
print("STAGE 27 | SOURCE-GROUNDED RESEARCH SYNTHESIS QUALITY GATE: PASSED")
print("="*76)
print("Stages 23–26 executed evidence and source lineage: VERIFIED")
print("Seven Development comparison models: INCLUDED")
print("Thirty-six Stage 24 robustness jobs: SUMMARISED")
print("Two frozen Stage 26 models and both reserved people: INCLUDED")
print("Rare-class support and limitations: EXPLICIT")
print("Raw HARTH data loaded / new model fitting / Test re-scoring: NO")
print("Outputs stored:", out)
print("Next: save/commit executed Stage 27 and review before GitHub release")
print("="*76)